# Residual CNN -- Complementing a Linear Reconstructor

The classical linear reconstructor (`WFS.BuildReconstructionMatrix`/`GetReconstructedOPD`, `AI4AO/TorchPropagator.py`) is already a strong, well-understood baseline -- it is exactly the machinery `TwinCalibrator` fits against real bench data. But it is fundamentally limited to whatever is linear in the WFS signal. This notebook trains a **residual CNN**: a frozen, pre-calibrated linear reconstructor supplies a baseline modal-coefficient estimate, and a small CNN is trained end-to-end, through the same physically meaningful closed-loop loss as every other reconstructor in this repo, to predict *only the linear reconstructor's error* -- added to, not replacing, the linear estimate. See `Ideas/10-residual-cnn-linear-complement.md` for the full motivation and design discussion.

**Three-way ablation:**
1. **Linear-only** -- the frozen linear reconstructor alone, never trained; the reference floor.
2. **CNN-only** -- a `PupilCNN` trained from scratch, no linear term at all (`Ideas/07-cnn-architecture-comparison.md`'s "stem/encoder" architecture, trained here with the ordinary `Trainer`).
3. **Linear + Residual CNN** -- this idea: a second `PupilCNN`, trained to predict the linear reconstructor's residual error.

**Implementation approach:** rather than wrapping the linear reconstructor and the CNN in a new composite `nn.Module`, condition 3 is trained with a hand-rolled loop -- a modified copy of `Trainer.train()`/`Trainer.evaluate()` with the linear reconstruction computed inline (mirroring `DualSensorFusion.ipynb`'s `train_fused`/`evaluate_fused`). `Trainer` only ever hands a reconstructor the *preprocessed* pupil-image stack, never the raw WFS frame the linear reconstructor's calibration needs -- a hand-rolled loop has that raw frame directly in scope, so it needs no new calibration machinery at all.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os
from tqdm import tqdm

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.Trainer import EvaluationResult
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Configuration and instrument

`ResidualCNNReconstructor_params.py` holds one `WFSParams`/`AtmosParams`/`LoopParams`/`DMParams`/`TrainParams` for a single fictional-demo, nominal-geometry modulated Pyramid WFS + DM -- no real bench behind it, deliberately identical to `CNNArchitectureComparison_params.py`'s instrument. `wfs` and `dm` are built fresh and frozen (`.eval()` + `dm.requires_grad_(False)`), since only the two trained CNNs' weights are ever optimized here.

In [ ]:
paramfile = 'ResidualCNNReconstructor_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = "../../Data/ResidualCNNReconstructor/"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")

## Calibrating the linear reconstructor

`M2C = dm.MakeZernikeM2C()` is the same Zernike modal command basis every reconstructor in this repo targets. Rather than building a separate Zernike basis to calibrate the linear reconstructor against (which risks drifting out of sync with `M2C` -- e.g. by using the WFS's pupil instead of the DM's own), `modes = dm(M2C.T)` pushes `M2C` itself through the DM's influence-function model and returns the *actual physical OPD* the DM produces for each mode -- already in meters, no separate wavenumber scaling needed. That is exactly the same computation every existing notebook already does to build `z_inv`, reused here for a second purpose: calibrating `wfs`'s reconstruction matrix (`wfs.BuildReconstructionMatrix(modes)`) against a basis that is, by construction, identical to `M2C`'s columns. `wfs`/`dm` are frozen, so this calibration only needs to happen once, before any training starts.

In [ ]:
M2C = dm.MakeZernikeM2C()
modes = dm(M2C.T)  # (Nmodes, Nres, Nres): the actual OPD the DM produces for each M2C column
z_inv = torch.linalg.pinv(modes.flatten(start_dim=-2))

with torch.no_grad():
    wfs.BuildReferenceIntensity()
    wfs.BuildReconstructionMatrix(modes)

print(f"Linear reconstructor calibrated: reconstructionMatrix {tuple(wfs.reconstructionMatrix.shape)}")

## `PupilCNN` architecture

The stem/encoder/head architecture from `DualSensorFusion.ipynb`, reused verbatim (this repo's convention: each notebook keeps its own copy rather than importing from the largely-unused `AI4AO/PhaseEstimators.py`). Each pupil image is processed independently in a `groups=n_channels` stem before a shared encoder mixes information across pupils. The exact same class is used for both the CNN-only baseline and the residual branch inside the Linear+Residual-CNN condition -- only the training procedure differs between the two.

In [ ]:
class PupilCNN(nn.Module):
    def __init__(self, n_channels, Nmodes):
        super().__init__()

        self.stem = nn.Sequential(
            # Process each pupil image independently
            nn.Conv2d(n_channels, 8 * n_channels, kernel_size=11, padding=5, groups=n_channels),
            nn.GELU(),

            nn.Conv2d(8 * n_channels, 16 * n_channels, kernel_size=7, padding=3, groups=n_channels),
            nn.GELU(),

            nn.MaxPool2d(2),
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(16 * n_channels, 64, kernel_size=5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, Nmodes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

## Condition 2: CNN-only

A plain `PupilCNN`, trained from scratch with no linear term at all, via the ordinary, unmodified `Trainer` -- exactly `Ideas/07`'s architecture #2. This is the "no linear prior" arm of the ablation.

In [ ]:
cnn_only = PupilCNN(n_channels=n_channels, Nmodes=Nmodes).to(device=device)
total_params = sum(p.numel() for p in cnn_only.parameters() if p.requires_grad)
print(f"CNN-only reconstructor -- total trainable parameters: {total_params:,}")

optimizer_cnn_only = torch.optim.AdamW(cnn_only.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

trainer_cnn_only = Trainer(
    wfs=wfs,
    framePreprocessor=framePreprocessor,
    dm=dm,
    M2C=M2C,
    phaseReconstructor=cnn_only,
    dataset=dataset,
    loss=loss,
    optimizer=optimizer_cnn_only,
)

try:
    trainer_cnn_only.load_checkpoint(PATH + "CNNOnly.pth", load_optimizer=False)
except (FileNotFoundError, RuntimeError):
    print("Starting from scratch")

In [ ]:
loss_tracker_cnn_only, loss_tracker_cnn_only_ideal = trainer_cnn_only.train(
    TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations']
)

In [ ]:
trainer_cnn_only.save_checkpoint(PATH + "CNNOnly.pth")

## Hand-rolled training for condition 3: Linear + Residual CNN

`Trainer.train()`/`Trainer.evaluate()` only ever call `self.phaseReconstructor(preprocessed_frames)` -- the reconstructor never sees the raw WFS frame the linear branch's `GetReconstructedOPD` needs. `train_residual`/`evaluate_residual` below are modified copies of `Trainer.train()`/`Trainer.evaluate()` (mirroring `DualSensorFusion.ipynb`'s `train_fused`/`evaluate_fused`: they close over `wfs`/`dm`/`framePreprocessor`/`M2C`/`z_inv` as notebook-level objects, and take `dataset`/the CNN/optimizer/loss explicitly) that compute the frozen linear estimate inline, from the raw `wfs_frames` already in scope, and add the CNN's own output to it:

```python
z_linear = wfs.GetReconstructedOPD(wfs_frames)   # frozen, no grad
z_output = z_linear + cnn(preprocessed_frames)    # was: z_output = self.phaseReconstructor(preprocessed_frames)
```

This is a deliberate trade-off: some code is duplicated between these functions and `Trainer.train()`/`evaluate()`, in exchange for a training loop that stays simple to read, debug, and later port into a real-instrument control script (see `Ideas/10`'s "Known risks" for the full discussion). `train_residual` also tracks a third quantity alongside the usual `total_loss`/`ideal_loss`: `linear_loss`, the loss the linear reconstructor alone would achieve at each step (computed the same no-grad way `ideal_loss` already is) -- shown live in the progress bar, so the CNN's actual contribution over the linear baseline is visible throughout training, not just in the final comparison.

In [ ]:
def train_residual(training_steps, closed_loop_iterations, dataset, cnn, optimizer, loss):
    """Modified copy of Trainer.train() (AI4AO/Trainer.py) that adds the frozen linear
    reconstructor's estimate to cnn's own output before it reaches the DM, so cnn only
    has to learn the linear reconstructor's residual error. Closes over wfs/dm/
    framePreprocessor/M2C/z_inv as notebook-level objects, mirroring DualSensorFusion.
    ipynb's train_fused."""
    if dm.training:
        dm.eval()
    if wfs.training:
        wfs.eval()

    loss_tracker = torch.zeros(training_steps // closed_loop_iterations, device=device)
    loss_tracker_linear = torch.zeros(training_steps // closed_loop_iterations, device=device)
    loss_tracker_ideal = torch.zeros(training_steps // closed_loop_iterations, device=device)

    cnn.train()

    M2C_T = M2C.T
    Nmodes = z_inv.shape[-1]

    progressBar = tqdm(range(training_steps // closed_loop_iterations))

    for u in progressBar:
        with torch.no_grad():
            batch = dataset[0]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]
            gain = batch["loop_gain"]
            leak = batch["loop_leak"]
            photons = batch["nphotons"]
            ron = batch["ron"]

            wfs.SetPhotonsAndRON(photons, ron)

            z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
            z_buffer = torch.zeros_like(z_estimated)
            z_output = torch.zeros_like(z_estimated)
            opd_reconstructed = torch.zeros_like(opd_gt)

            total_loss = 0
            linear_loss = 0
            ideal_loss = 0

        for i in range(closed_loop_iterations):
            with torch.no_grad():
                if i > 0:
                    batch = dataset[i]
                    opd_gt = batch["opd"]
                    pupilGT = batch["pupil"]

                residual_opd = opd_gt - opd_reconstructed
                Ze = torch.matmul(residual_opd.flatten(start_dim=-2), z_inv)

                z_estimated = z_estimated * leak + gain * z_buffer
                z_buffer = torch.clone(z_output)

                wfs_frames = wfs(residual_opd, pupilGT)
                preprocessed_frames = framePreprocessor.ProcessFrame(wfs_frames)
                z_linear = wfs.GetReconstructedOPD(wfs_frames)

            z_output = z_linear + cnn(preprocessed_frames)

            opd_reconstructed = dm(z_estimated @ M2C_T)
            opd_reconstructed_iter = dm(z_output @ M2C_T)
            opd_reconstructed_iter_ideal = dm(Ze @ M2C_T)

            corrected_residual_opd = residual_opd - opd_reconstructed_iter
            total_loss = total_loss + loss(Ze, z_output, pupilGT, residual_opd, corrected_residual_opd, wfs_frames) / closed_loop_iterations

            with torch.no_grad():
                opd_reconstructed_iter_linear = dm(z_linear @ M2C_T)
                corrected_residual_opd_linear = residual_opd - opd_reconstructed_iter_linear
                linear_loss = linear_loss + loss(Ze, z_linear, pupilGT, residual_opd, corrected_residual_opd_linear, wfs_frames) / closed_loop_iterations

                ideal_corrected_residual_opd = residual_opd - opd_reconstructed_iter_ideal
                ideal_loss = ideal_loss + loss(Ze, Ze, pupilGT, residual_opd, ideal_corrected_residual_opd, wfs_frames) / closed_loop_iterations

        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        optimizer.step()

        loss_tracker[u] = total_loss.detach()
        loss_tracker_linear[u] = linear_loss.detach()
        loss_tracker_ideal[u] = ideal_loss.detach()

        if u % (300 // closed_loop_iterations) == 1:
            lower_lim = max(0, u - 100 // closed_loop_iterations)
            progressBar.set_postfix({
                'Loss': float(loss_tracker[lower_lim:u].mean()),
                'Loss_linear': float(loss_tracker_linear[lower_lim:u].mean()),
                'Loss_ideal': float(loss_tracker_ideal[lower_lim:u].mean()),
            })

    return loss_tracker, loss_tracker_linear, loss_tracker_ideal

In [ ]:
@torch.no_grad()
def evaluate_residual(n_steps, dataset, cnn=None, gain=None, leak=None, psf_sampling=4, psf_fov=20):
    """Modified copy of Trainer.evaluate() that adds the frozen linear reconstructor's
    estimate to cnn's output, with no backprop and no pupil noise. Passing cnn=None
    evaluates the linear reconstructor alone (the Linear-only condition)."""
    if dm.training:
        dm.eval()
    if wfs.training:
        wfs.eval()
    if cnn is not None:
        cnn.eval()

    M2C_T = M2C.T
    Nmodes = z_inv.shape[-1]

    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = gain if gain is not None else batch["loop_gain"]
    leak = leak if leak is not None else batch["loop_leak"]
    photons = batch["nphotons"]
    ron = batch["ron"]

    wfs.SetPhotonsAndRON(photons, ron)

    z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
    z_buffer = torch.zeros_like(z_estimated)
    z_output = torch.zeros_like(z_estimated)
    opd_reconstructed = torch.zeros_like(opd_gt)

    opds, pupils, reconstructed, residuals, frames, psfs, outputs = [], [], [], [], [], [], []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed

        wfs_frames = wfs(residual_opd, pupilGT)
        psf = wfs.GetPSF(residual_opd, pupilGT, psf_sampling, psf_fov)
        preprocessed_frames = framePreprocessor.ProcessFrame(wfs_frames, False)
        z_linear = wfs.GetReconstructedOPD(wfs_frames)
        z_output = z_linear + (cnn(preprocessed_frames) if cnn is not None else 0.0)

        z_buffer = torch.clone(z_output)

        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_buffer

        opd_reconstructed = dm(z_estimated @ M2C_T)

        opds.append(opd_gt)
        pupils.append(pupilGT)
        reconstructed.append(opd_reconstructed)
        residuals.append(residual_opd)
        frames.append(wfs_frames)
        psfs.append(psf)
        outputs.append(z_output)

    return EvaluationResult(
        opd=torch.stack(opds),
        pupil=torch.stack(pupils),
        opd_reconstructed=torch.stack(reconstructed),
        residual_opd=torch.stack(residuals),
        wfs_frames=torch.stack(frames),
        psfs=torch.stack(psfs),
        z_output=torch.stack(outputs),
    )

In [ ]:
def save_residual_checkpoint(path, model, optimizer):
    torch.save({
        "phase_reconstructor_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, path)


def load_residual_checkpoint(path, model, optimizer, load_optimizer=True):
    if not os.path.exists(path):
        print(f"No checkpoint found at {path}, starting from scratch")
        return
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["phase_reconstructor_state_dict"])
    if load_optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

## Condition 3: training

A second `PupilCNN` (standard init -- no zero-init trick, per this session's decision), trained via `train_residual` to predict the linear reconstructor's residual error.

In [ ]:
residual_cnn = PupilCNN(n_channels=n_channels, Nmodes=Nmodes).to(device=device)
total_params = sum(p.numel() for p in residual_cnn.parameters() if p.requires_grad)
print(f"Residual CNN -- total trainable parameters: {total_params:,}")

optimizer_residual = torch.optim.AdamW(residual_cnn.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

load_residual_checkpoint(PATH + "LinearPlusResidual.pth", residual_cnn, optimizer_residual, load_optimizer=False)

In [ ]:
loss_tracker_residual, loss_tracker_residual_linear, loss_tracker_residual_ideal = train_residual(
    TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'], dataset, residual_cnn, optimizer_residual, loss
)

In [ ]:
save_residual_checkpoint(PATH + "LinearPlusResidual.pth", residual_cnn, optimizer_residual)

## Comparing training-loss curves

`Trainer.plot_losses` only overlays two trackers at a time; condition 3 now tracks three (`total`, `linear-only`, `ideal`). All four curves below share the same physical units, so they can be overlaid directly: condition 3's own total loss, its `linear_loss` reference (what the linear reconstructor alone would achieve at each step), condition 2's CNN-only loss, and the oracle "ideal loss" bound (a property of the atmosphere/noise/DM, not of any reconstructor).

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")


fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(smooth(loss_tracker_residual), label="Linear + Residual CNN")
ax.plot(smooth(loss_tracker_residual_linear), label="Linear-only (reference)", linestyle="--")
ax.plot(smooth(loss_tracker_cnn_only), label="CNN-only")
ax.plot(smooth(loss_tracker_residual_ideal), label="Oracle bound (ideal)", color="black", linestyle=":")

ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.legend()
plt.show()

## Three-way closed-loop ablation

For each condition, reseed immediately before the rollout (`DualSensorFusion.ipynb`'s "reset the seed right before each rollout" trick) so all three see identical wind/r0/noise draws, then compute the steady-state residual wavefront error in nm RMS over the pupil, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()`/`evaluate_residual` use internally).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

results = {}
nm_rms = {}

torch.manual_seed(TrainParams['Seed'])
results["Linear-only"] = evaluate_residual(TrainParams['TestRunNb'], eval_dataset, cnn=None)

torch.manual_seed(TrainParams['Seed'])
results["CNN-only"] = trainer_cnn_only.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)

torch.manual_seed(TrainParams['Seed'])
results["Linear+Residual"] = evaluate_residual(TrainParams['TestRunNb'], eval_dataset, cnn=residual_cnn)

for name, result in results.items():
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:16s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")

In [ ]:
param_counts = {
    "Linear-only": 0,
    "CNN-only": sum(p.numel() for p in cnn_only.parameters() if p.requires_grad),
    "Linear+Residual": sum(p.numel() for p in residual_cnn.parameters() if p.requires_grad),
}

names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]
params_values = [param_counts[n] for n in names]

fig, axes = plt.subplots(1, 1, figsize=(6, 5))

axes.bar(names, rms_values)
axes.set_ylabel("Steady-state residual (nm RMS)")
axes.set_title("Closed-loop performance")

plt.tight_layout()
plt.show()

## Visualizing the best condition's closed loop

A closer look at whichever condition came out on top above -- the same rollout-animation pattern `05_TrainingAReconstructor.ipynb`/`Ideas/07`'s notebook use, reusing the rollout already computed above (no need to re-run).

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best condition: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

result = results[best_name]
n_frames = result.opd.shape[0]

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())